# saas_probe

> Free-by-default reachability/quota probes for Mistral, Datalab, and
> Replicate -- for answering "is it them or is it me" before trusting a
> `compare()` run against an engine that's behaving strangely.

`probe_mistral()`, `probe_datalab()`, and `probe_replicate()` each make a
minimal, mostly-free request against one engine's real API and return
status, latency, and whatever rate-limit information the response actually
carries. A quick check before a real comparison run: a failure here means
the problem is reachability or authentication, not the extraction itself.

Worth knowing before relying on any of these: a healthy-looking probe does
not guarantee the actual extraction endpoint is healthy. A lightweight
check like `GET /v1/models` can stay green while the real, heavier endpoint
(`POST /v1/ocr`, in Mistral's case) is failing -- the two are often served
by different infrastructure under the hood. `probe_mistral()`'s default
mode is deliberately labelled a reachability check, not an OCR-health
check, for exactly this reason -- see its docstring for the opt-in real
check.

In [ ]:
#| default_exp saas_probe

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import os
import time
from dataclasses import dataclass, field

import httpx

## `ProbeResult`

One shape for all three engines, even though the underlying checks are
different -- a models-list call for Mistral, a two-stage health+auth check
for Datalab, an account-info call for Replicate. What we actually want out
of each is the same: did it respond, how fast, and what (if anything) did
it tell us about our remaining quota.

In [ ]:
#| export
@dataclass
class ProbeResult:
    """Result of one reachability/quota probe against one engine's real API."""

    engine: str
    ok: bool
    status_code: int | None = None
    latency_s: float | None = None
    rate_limit_headers: dict[str, str] = field(default_factory=dict)
    detail: str | None = None

    def __str__(self) -> str:
        bits = [f"{self.engine}: {'ok' if self.ok else 'FAILED'}"]
        if self.status_code is not None:
            bits.append(f"HTTP {self.status_code}")
        if self.latency_s is not None:
            bits.append(f"{self.latency_s:.2f}s")
        if self.rate_limit_headers:
            bits.append(str(self.rate_limit_headers))
        if self.detail:
            bits.append(self.detail)
        return " | ".join(bits)

## Known limits (verify before relying on these)

SaaS providers change limits without notice -- treat the numbers below as a
starting point to verify with the probes in this notebook, not a permanent
fact.

| Engine | Tier | Documented limit | Known gap |
|---|---|---|---|
| Datalab | Team/paid | 400 req/min, 400 concurrent in-flight, 5,000 concurrent pages, 200MB / 7,000 pages per request | high concurrency (hundreds of simultaneous requests) not well characterised in practice |
| Datalab | Free | 10 req/min, concurrency 5 | -- |
| Replicate | account | ~600 req/min (approximate; not precisely documented) | **not the real bottleneck** -- see below |
| Mistral | account (scales with spend) | ~5 rps / ~300 req/min at entry tier, separate rps + tokens/min, per-model | concurrency behaviour and outage patterns are comparatively under-documented publicly |

Two things worth knowing that a rate-limit header alone won't tell us:

- **Replicate's real bottleneck is model-level concurrency, not the account
  rate limit.** A model running on [Cog](https://github.com/replicate/cog)
  (as Datalab's Marker model does, on Replicate) defaults to one prediction
  executing at a time *per model instance* -- unrelated to how many requests
  per minute the account is allowed to *submit*. Submitting five requests at
  once doesn't get five running in parallel; it gets one running and four
  queued. No amount of watching rate-limit headers reveals this -- it only
  shows up as requests taking successively longer to complete under load.
- **A healthy-looking probe doesn't mean the real endpoint is healthy.** A
  lightweight endpoint like `GET /v1/models` can return `200 OK` while a
  heavier, purpose-specific endpoint (`POST /v1/ocr`, for Mistral) is
  failing -- different infrastructure under the hood, especially for
  GPU-backed inference paths versus simple metadata lookups. A failure mode
  like this shows genuine backend processing time on the failing requests
  (not an instant rejection) and no rate-limit headers at all, ruling out
  quota exhaustion as the cause. `probe_mistral()` below defaults to the
  free, low-signal `/v1/models` check for exactly this reason: it is
  honestly labelled as a reachability check, not an OCR-health check.

## `probe_mistral()`

Two modes, because they answer different questions and cost different
amounts:

- **Default (`real_ocr_call=False`)**: `GET /v1/models`, free, fast --
  confirms the API key works and Mistral's API surface is reachable at all.
  A green result here is not proof the OCR endpoint itself is healthy (see
  above) -- it only confirms the lightweight metadata surface is up.
- **`real_ocr_call=True`**: submits one real, small OCR request (needs
  `pdf_path`) -- costs a fraction of a cent, and is the only way to see the
  OCR-specific rate-limit headers (`x-ratelimit-limit-ocr-pages-minute`
  and friends) or to actually exercise the endpoint that does the real work.

In [ ]:
#| export
def probe_mistral(
    api_key: str | None = None,
    real_ocr_call: bool = False,
    pdf_path: str | None = None,
) -> ProbeResult:
    """Probe Mistral's API. Free by default (GET /v1/models); pass
    real_ocr_call=True + pdf_path for a real (paid, small) OCR request that
    also surfaces OCR-specific rate-limit headers."""
    key = api_key or os.environ.get("MISTRAL_API_KEY", "")
    if not key:
        return ProbeResult(engine="mistral", ok=False, detail="MISTRAL_API_KEY not set")

    if not real_ocr_call:
        t0 = time.monotonic()
        try:
            r = httpx.get(
                "https://api.mistral.ai/v1/models",
                headers={"Authorization": f"Bearer {key}"}, timeout=10,
            )
            latency = time.monotonic() - t0
            headers = {k: v for k, v in r.headers.items() if "ratelimit" in k.lower() or "retry" in k.lower()}
            return ProbeResult(engine="mistral", ok=r.status_code == 200, status_code=r.status_code,
                                latency_s=latency, rate_limit_headers=headers,
                                detail=None if r.status_code == 200 else r.text[:200])
        except Exception as exc:
            return ProbeResult(engine="mistral", ok=False, detail=str(exc))

    if not pdf_path:
        raise ValueError("pdf_path is required when real_ocr_call=True")
    import base64
    pdf_b64 = base64.b64encode(open(pdf_path, "rb").read()).decode()
    t0 = time.monotonic()
    try:
        r = httpx.post(
            "https://api.mistral.ai/v1/ocr",
            headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
            json={
                "model": "mistral-ocr-latest",
                "document": {"type": "document_url", "document_url": f"data:application/pdf;base64,{pdf_b64}"},
                "include_image_base64": False,
                "pages": [0],
            },
            timeout=60,
        )
        latency = time.monotonic() - t0
        headers = {k: v for k, v in r.headers.items() if "ratelimit" in k.lower() or "retry" in k.lower()}
        return ProbeResult(engine="mistral", ok=r.status_code == 200, status_code=r.status_code,
                            latency_s=latency, rate_limit_headers=headers,
                            detail=None if r.status_code == 200 else r.text[:200])
    except Exception as exc:
        return ProbeResult(engine="mistral", ok=False, detail=str(exc))

## `probe_datalab()`

Datalab's public `/api/v1/health` endpoint returns `{"status": "ok"}`
regardless of whether the API key supplied elsewhere is valid -- it checks
whether Datalab is up, not whether *we* can use it. A real auth check needs
a second call: `GET /api/v1/convert` (a POST-only endpoint) returns **405**
for a valid key (method not allowed, but the key was accepted) and **401**
for an invalid one. Both stages are free -- neither one is an actual
conversion request.

In [ ]:
#| export
def probe_datalab(api_key: str | None = None) -> ProbeResult:
    """Two-stage probe: public /api/v1/health for reachability, then an
    authenticated GET /api/v1/convert to check the key (405 = valid key,
    401 = invalid). Both stages are free."""
    key = api_key or os.environ.get("DATALAB_API_KEY", "")
    if not key:
        return ProbeResult(engine="datalab", ok=False, detail="DATALAB_API_KEY not set")

    t0 = time.monotonic()
    try:
        health = httpx.get("https://www.datalab.to/api/v1/health", timeout=10)
    except Exception as exc:
        return ProbeResult(engine="datalab", ok=False, detail=f"health check failed: {exc}")
    if health.status_code != 200:
        return ProbeResult(engine="datalab", ok=False, status_code=health.status_code,
                            latency_s=time.monotonic() - t0, detail="service unreachable")

    try:
        auth = httpx.get(
            "https://www.datalab.to/api/v1/convert",
            headers={"X-Api-Key": key}, timeout=10,
        )
    except Exception as exc:
        return ProbeResult(engine="datalab", ok=False, detail=f"auth check failed: {exc}")
    latency = time.monotonic() - t0
    headers = {k: v for k, v in auth.headers.items() if "ratelimit" in k.lower() or "retry" in k.lower()}
    if auth.status_code in (405, 404):
        return ProbeResult(engine="datalab", ok=True, status_code=auth.status_code,
                            latency_s=latency, rate_limit_headers=headers, detail="key accepted")
    if auth.status_code in (401, 403):
        return ProbeResult(engine="datalab", ok=False, status_code=auth.status_code,
                            latency_s=latency, detail="key rejected")
    return ProbeResult(engine="datalab", ok=False, status_code=auth.status_code,
                        latency_s=latency, detail=auth.text[:200])

## `probe_replicate()`

`GET /v1/account` -- confirms the token is valid and the API is reachable,
free. It cannot reveal Replicate's real bottleneck for this use case (the
model-level concurrency=1 limit noted above): that only shows up as queued
predictions taking longer under load, not as anything visible in a
single account-info call.

In [ ]:
#| export
def probe_replicate(api_token: str | None = None) -> ProbeResult:
    """Probe Replicate via GET /v1/account. Free. Does not reveal the
    model-level concurrency limit -- see the module docs above."""
    token = api_token or os.environ.get("REPLICATE_API_TOKEN", "")
    if not token:
        return ProbeResult(engine="replicate", ok=False, detail="REPLICATE_API_TOKEN not set")
    t0 = time.monotonic()
    try:
        r = httpx.get(
            "https://api.replicate.com/v1/account",
            headers={"Authorization": f"Token {token}"}, timeout=10,
        )
        latency = time.monotonic() - t0
        headers = {k: v for k, v in r.headers.items() if "ratelimit" in k.lower() or "retry" in k.lower()}
        return ProbeResult(engine="replicate", ok=r.status_code == 200, status_code=r.status_code,
                            latency_s=latency, rate_limit_headers=headers,
                            detail=None if r.status_code == 200 else r.text[:200])
    except Exception as exc:
        return ProbeResult(engine="replicate", ok=False, detail=str(exc))

## `probe_all()`

Runs whichever of the three have a key/token in the environment, skips the
rest, and returns every result -- all free-tier checks by default (Mistral
stays on the `/v1/models` check unless told otherwise).

In [ ]:
#| export
def probe_all(real_mistral_ocr_call: bool = False, pdf_path: str | None = None) -> list[ProbeResult]:
    """Run probe_mistral/probe_datalab/probe_replicate for whichever engines
    have credentials in the environment. Free-tier checks only unless
    real_mistral_ocr_call=True."""
    results = []
    if os.environ.get("MISTRAL_API_KEY"):
        results.append(probe_mistral(real_ocr_call=real_mistral_ocr_call, pdf_path=pdf_path))
    if os.environ.get("DATALAB_API_KEY"):
        results.append(probe_datalab())
    if os.environ.get("REPLICATE_API_TOKEN"):
        results.append(probe_replicate())
    return results

### Try it

All three calls below are free (no OCR/conversion request made) and real --
no mocking. Run this before trusting a `compare()`/`estimate_overhead()`
run against an engine that's behaving strangely: `ok=False` here means the
problem is reachability or auth, not the extraction itself; `ok=True` does
not guarantee the actual conversion endpoint is healthy, just that the
surface API is up (see "Known limits" above).

In [ ]:
#| eval: false
for r in probe_all():
    print(r)

---
Three free checks, each answering a narrower question than it might look
like at first: reachable, authenticated, and (where the response provides
it) how much quota is left -- not "is the real extraction endpoint
healthy," which needs the opt-in real call in `probe_mistral()` or an
actual `compare()` run.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()